### 3.2 einstein examples

In [29]:
import torch
from einops import einsum, rearrange
import einx

images = torch.randn(64, 128, 128, 3)
vec = torch.linspace(start=0, end=1, steps=10)
print(images.shape, vec.shape)
ve = rearrange(vec, "D -> 1 D 1 1 1")
imgs = rearrange(images, "B H W C -> B 1 H W C")
ima_vec = imgs * ve
print(ima_vec.shape)
img_vec = einsum(images, vec, "B H W C, D -> B D H W C")
print(img_vec.shape)

torch.Size([64, 128, 128, 3]) torch.Size([10])
torch.Size([64, 10, 128, 128, 3])
torch.Size([64, 10, 128, 128, 3])


#### 原生pytorch写法

In [ ]:
images = torch.randn(64, 32, 32, 3)
transform = torch.randn(32 * 32, 32 * 32)
# 展平宽和高，然后交换维度
images1 = images.view(-1, images.size(1) * images.size(2), images.size(3))
images2 = images1.transpose(1, 2)
# 应用线形变换
images3 = images2 @ transform.T
# 还原交换维度，再恢复原始图像
images4 = images3.transpose(1, 2)
images5 = images4.view(*images.size())
print(f"images.shape{images.shape}")
print(f"images1.shape{images1.shape}")
print(f"images2.shape{images2.shape}")
print(f"images3.shape{images3.shape}")
print(f"images4.shape{images4.shape}")
print(f"images5.shape{images5.shape}")

images.shapetorch.Size([64, 32, 32, 3])
images1.shapetorch.Size([64, 1024, 3])
images2.shapetorch.Size([64, 3, 1024])
images3.shapetorch.Size([64, 3, 1024])
images4.shapetorch.Size([64, 1024, 3])
images5.shapetorch.Size([64, 32, 32, 3])


#### 改用einops

In [25]:
images = torch.randn(64, 32, 32, 3)
H = W = 32
transform = torch.randn(32 * 32, 32 * 32)
# 展平宽和高，然后交换维度
images1 = rearrange(images, "B H W C -> B C (H W)")
# 应用线形变换
images2 = einsum(images1, transform, "B C t_in, t_out t_in -> B C t_out")
# 还原交换维度，再恢复原始图像
images3 = rearrange(images2, "B C (H W) -> B H W C", H=H, W=W)
print(f"images.shape{images.shape}")
print(f"images1.shape{images1.shape}")
print(f"images2.shape{images2.shape}")
print(f"images3.shape{images3.shape}")

images.shapetorch.Size([64, 32, 32, 3])
images1.shapetorch.Size([64, 3, 1024])
images2.shapetorch.Size([64, 3, 1024])
images3.shapetorch.Size([64, 32, 32, 3])


### 再用einx.dot

In [31]:
H = W = 32
images = torch.randn(64, 32, 32, 3)
transform = torch.randn(32 * 32, 32 * 32)
images1 = einx.dot(
    "B H_in W_in C, (H_out W_out) (H_in W_in) -> B H_out W_out C",
    images,
    transform,
    H_out=H,
    W_out=W
)
print(f"images.shape{images.shape}")
print(f"images1.shape{images1.shape}")

images.shapetorch.Size([64, 32, 32, 3])
images1.shapetorch.Size([64, 32, 32, 3])
